# Experience 1 : quantification (bf16 / fp16 / INT8 / INT4) - Qwen2.5-Coder-1.5B

Meme benchmark que `benchmark_<modele>.ipynb` (15 prompts + 20 problemes HumanEval, decodage glouton),
repete pour 4 variantes de precision du **meme** modele :

| Variante | Description |
|---|---|
| `bf16` | reference de l'equipe (deja mesuree, on la refait pour verifier la reproductibilite) |
| `fp16` | la precision citee dans le sujet |
| `int8` | quantification 8 bits (bitsandbytes), calcul en fp16 |
| `nf4`  | quantification 4 bits NF4 + double quantification (bitsandbytes), calcul en fp16 |

Avant de lancer : T4 GPU + Internet actives. Resultats dans `results/quant/` (n'affecte pas `merge_results.py`).


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH = "main"          # branche ou le groupe a fusionne les resultats des 4 modeles

if not os.path.exists("../config.json"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results/quant", exist_ok=True)

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 231 (delta 102), reused 86 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 171.71 KiB | 13.21 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [2]:
!pip install -r ../requirements.txt -q
!pip install -q -U bitsandbytes
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-fa03a6ac-d496-4d7a-d4be-2b4b5f1d68b1)
GPU 1: Tesla T4 (UUID: GPU-51084266-6d64-82a2-3b22-3fd279274572)


In [3]:
import json, time, sys, re, gc, platform, shutil
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

MODEL_KEY = "qwen"
VARIANTS = ["bf16", "fp16", "int8", "nf4"]     # ordre d'execution
OVERWRITE = False                               # False : saute une variante deja terminee (reprise apres crash)

with open("../config.json") as f:
    config = json.load(f)
MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
OUT = "../results/quant"
print(MODEL_NAME, "|", GPU_NAME)

def norm(s):
    return re.sub(r"\s+", "", str(s))

# Reference bf16 deja publiee par le groupe (sert a comparer les reponses et a recopier les notes manuelles)
BASE_A_PATH = f"../results/benchmark_{model_slug}.csv"
BASE_B_PATH = f"../results/humaneval_{model_slug}.json"
base_a = pd.read_csv(BASE_A_PATH).set_index("id") if os.path.exists(BASE_A_PATH) else None
base_b = ({d["task_id"]: d["status"] for d in json.load(open(BASE_B_PATH))["details"]}
          if os.path.exists(BASE_B_PATH) else None)
print("Reference bf16 :", "trouvee" if base_a is not None else "ABSENTE (same_as_bf16 sera faux partout)")

Qwen/Qwen2.5-Coder-1.5B-Instruct | Tesla T4
Reference bf16 : trouvee


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model(variant):
    kw = {"device_map": {"": 0}}
    if variant == "bf16":
        kw["dtype"] = torch.bfloat16
    elif variant == "fp16":
        kw["dtype"] = torch.float16
    elif variant == "int8":
        kw["dtype"] = torch.float16
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif variant == "nf4":
        kw["dtype"] = torch.float16
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        raise ValueError(variant)
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw)
    m.eval()
    return m

def generate(model, user_prompt):
    # IDENTIQUE a la fonction du benchmark de base (memes mesures), sauf que le modele est passe en argument
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]
with open("../humaneval_subset.json") as f:
    subset = json.load(f)
humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
HEADER = "from typing import *\nimport math\n\n"
LABEL_COLS = ("bug_label", "note_qualite", "hallucination")

def run_part_a(model, variant):
    rows = []
    for p in prompts:
        gen = generate(model, p["prompt"])
        response = gen.pop("response")
        status, error = None, None
        if p["task"] == "generation_code" and "test" in p:
            status, error = run_tests(extract_code(response), p["test"], timeout=HE_CONFIG["timeout_seconds"])
        elif p["task"] == "completion_code" and "expected" in p:
            status = "pass" if norm(p["expected"]) in norm(response) else "fail"
        same = bool(base_a is not None and p["id"] in base_a.index
                    and norm(base_a.loc[p["id"], "response"]) == norm(response))
        row = {"variant": variant, "id": p["id"], "task": p["task"], "prompt": p["prompt"],
               "response": response, **gen, "auto_status": status, "auto_error": error,
               "same_as_bf16": same}
        # meme texte que la reference -> meme note (on recopie) ; sinon a noter a la main
        for c in LABEL_COLS:
            row[c] = base_a.loc[p["id"], c] if same else None
        rows.append(row)
        print(variant, p["id"], status, "| identique bf16:", same)
    return pd.DataFrame(rows)

def run_part_b(model, variant):
    details = []
    for ex in humaneval:
        gen = generate(model, HE_CONFIG["instruction"] + ex["prompt"])
        code_generated = extract_code(gen["response"])
        test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
        status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                                  timeout=HE_CONFIG["timeout_seconds"])
        details.append({"task_id": ex["task_id"], "status": status, "error": error,
                        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
                        "tokens_per_sec": gen["tokens_per_sec"], "output_tokens": gen["output_tokens"],
                        "gpu_memory_go": gen["gpu_memory_go"], "response": gen["response"]})
        print(variant, ex["task_id"], status)
    n_pass = sum(d["status"] == "pass" for d in details)
    return {"model": MODEL_NAME, "variant": variant, "gpu": GPU_NAME, "n_problems": len(details),
            "n_pass": n_pass, "pass@1": round(100 * n_pass / len(details), 1), "details": details}

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

In [6]:
def summarize(variant, df_a, he, weights_gb):
    auto = df_a[df_a["auto_status"].notna()]
    he_tps = sum(d["tokens_per_sec"] for d in he["details"]) / len(he["details"])
    he_agree = (sum(base_b.get(d["task_id"]) == d["status"] for d in he["details"]) / len(he["details"]) * 100
                if base_b else None)
    return {
        "variant": variant,
        "weights_gb": round(weights_gb, 2),
        "peak_gpu_gb": df_a["gpu_memory_go"].max(),
        "ttft_s": round(df_a["ttft_s"].mean(), 3),
        "latency_s": round(df_a["latency_s"].mean(), 2),
        "tokens_per_sec": round(df_a["tokens_per_sec"].mean(), 2),
        "output_tokens": round(df_a["output_tokens"].mean(), 1),
        "auto_pass_P1_P6": f"{(auto['auto_status'] == 'pass').sum()}/{len(auto)}",
        "humaneval_pass@1_%": he["pass@1"],
        "he_tokens_per_sec": round(he_tps, 2),
        "he_same_status_as_bf16_%": he_agree,
        "same_text_as_bf16_%": round(df_a["same_as_bf16"].mean() * 100, 1),
        "rows_to_grade_by_hand": int((~df_a["same_as_bf16"]).sum()),
    }

for variant in VARIANTS:
    tag = f"{model_slug}__{variant}"
    if not OVERWRITE and os.path.exists(f"{OUT}/summary_{tag}.json"):
        print(f"== {variant} deja fait, on saute ==")
        continue
    print(f"\n========== {variant} ==========")
    model = load_model(variant)
    weights_gb = model.get_memory_footprint() / 1e9
    print(f"Taille des poids en memoire : {weights_gb:.2f} Go")

    # echauffement + controle que le modele repond vraiment
    t = generate(model, "Write a Python function add(a, b) that returns a + b.")
    print(t["response"][:200])
    assert "def" in t["response"], f"Reponse incoherente avec la variante {variant}"

    df_a = run_part_a(model, variant)
    he = run_part_b(model, variant)
    df_a.to_csv(f"{OUT}/benchmark_{tag}.csv", index=False)
    with open(f"{OUT}/humaneval_{tag}.json", "w", encoding="utf-8") as f:
        json.dump(he, f, ensure_ascii=False, indent=2)
    with open(f"{OUT}/summary_{tag}.json", "w") as f:
        json.dump(summarize(variant, df_a, he, weights_gb), f, indent=2)

    del model
    gc.collect(); torch.cuda.empty_cache()
print("\nTermine.")


========== bf16 ==========


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Taille des poids en memoire : 3.09 Go
```python
def add(a, b):
    return a + b
```

In this solution, the function `add` takes two parameters, `a` and `b`, and returns their sum. The function is straightforward and utilizes Python's buil
bf16 P1 pass | identique bf16: True
55
bf16 P2 pass | identique bf16: True
monde le Bonjour
bf16 P3 pass | identique bf16: True
3
bf16 P4 pass | identique bf16: True
bf16 P5 fail | identique bf16: True
bf16 P6 pass | identique bf16: True
bf16 P7 None | identique bf16: True
bf16 P8 None | identique bf16: True
bf16 P9 None | identique bf16: True
bf16 P10 None | identique bf16: True
bf16 P11 None | identique bf16: True
bf16 P12 None | identique bf16: True
bf16 P13 None | identique bf16: True
bf16 P14 None | identique bf16: True
bf16 P15 None | identique bf16: True
bf16 HumanEval/0 pass
bf16 HumanEval/1 fail
bf16 HumanEval/2 pass
bf16 HumanEval/3 pass
bf16 HumanEval/4 pass
bf16 HumanEval/5 pass
bf16 HumanEval/6 fail
bf16 HumanEval/7 pass
bf16 HumanEval/8 

... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Taille des poids en memoire : 1.78 Go
```python
def add(a, b):
    return a + b
```

In this solution, the function `add` takes two parameters, `a` and `b`, and returns their sum. The function is straightforward and utilizes Python's buil
int8 P1 pass | identique bf16: False
5
int8 P2 pass | identique bf16: False
universe the from world Hello
int8 P3 pass | identique bf16: False
3
int8 P4 pass | identique bf16: False
int8 P5 fail | identique bf16: False
int8 P6 pass | identique bf16: False
int8 P7 None | identique bf16: False
int8 P8 None | identique bf16: False
int8 P9 None | identique bf16: False
int8 P10 None | identique bf16: False
int8 P11 None | identique bf16: False
int8 P12 None | identique bf16: False
int8 P13 None | identique bf16: False
int8 P14 None | identique bf16: False
int8 P15 None | identique bf16: False
int8 HumanEval/0 pass
int8 HumanEval/1 pass
int8 HumanEval/2 pass
int8 HumanEval/3 pass
int8 HumanEval/4 pass
int8 HumanEval/5 pass
int8 HumanEval/6 fail
int8 HumanEv

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Taille des poids en memoire : 1.12 Go
```python
def add(a, b):
    return a + b
```

In this solution, the function `add` takes two parameters, `a` and `b`, which represent the numbers to be added together. The function then returns the s
nf4 P1 pass | identique bf16: False
55
nf4 P2 pass | identique bf16: False
monde le Bonjour
nf4 P3 pass | identique bf16: False
3
nf4 P4 pass | identique bf16: False
nf4 P5 fail | identique bf16: False
nf4 P6 pass | identique bf16: False
nf4 P7 None | identique bf16: False
nf4 P8 None | identique bf16: False
nf4 P9 None | identique bf16: False
nf4 P10 None | identique bf16: False
nf4 P11 None | identique bf16: False
nf4 P12 None | identique bf16: False
nf4 P13 None | identique bf16: False
nf4 P14 None | identique bf16: False
nf4 P15 None | identique bf16: False
nf4 HumanEval/0 pass
nf4 HumanEval/1 pass
nf4 HumanEval/2 pass
nf4 HumanEval/3 pass
nf4 HumanEval/4 pass
nf4 HumanEval/5 pass
nf4 HumanEval/6 fail
nf4 HumanEval/7 pass
nf4 HumanEval/8 pass
nf4 

In [7]:
rows = []
for variant in VARIANTS:
    path = f"{OUT}/summary_{model_slug}__{variant}.json"
    if os.path.exists(path):
        rows.append(json.load(open(path)))
table = pd.DataFrame(rows).set_index("variant")
if "fp16" in table.index:
    table["speed_vs_fp16"] = (table["tokens_per_sec"] / table.loc["fp16", "tokens_per_sec"]).round(2)
    table["peak_mem_vs_fp16"] = (table["peak_gpu_gb"] / table.loc["fp16", "peak_gpu_gb"]).round(2)
table.to_csv(f"{OUT}/comparatif_quantification_{model_slug}.csv")
print(f"-> {OUT}/comparatif_quantification_{model_slug}.csv")
table.T

-> ../results/quant/comparatif_quantification_qwen2.5-coder-1.5b-instruct.csv


variant,bf16,fp16,int8,nf4
weights_gb,3.09,3.09,1.78,1.12
peak_gpu_gb,3.12,3.12,1.81,1.2
ttft_s,0.11,0.048,0.196,0.066
latency_s,8.87,8.4,35.16,12.41
tokens_per_sec,25.85,26.11,6.59,19.01
output_tokens,229.6,219.3,231.9,235.9
auto_pass_P1_P6,5/6,5/6,5/6,5/6
humaneval_pass@1_%,80.0,85.0,85.0,85.0
he_tokens_per_sec,25.27,25.99,6.63,19.05
he_same_status_as_bf16_%,100.0,95.0,95.0,95.0


In [8]:
from importlib.metadata import version
lines = [f"# Environnement - quantification {MODEL_NAME}", "", f"- GPU : {GPU_NAME}",
         f"- Python : {platform.python_version()}", f"- torch : {torch.__version__}", f"- CUDA : {torch.version.cuda}"]
for lib in ["transformers", "accelerate", "bitsandbytes", "huggingface_hub", "datasets", "pandas"]:
    try:
        lines.append(f"- {lib} : {version(lib)}")
    except Exception:
        lines.append(f"- {lib} : non installe")
open(f"{OUT}/environnement_quant_{model_slug}.md", "w").write("\n".join(lines) + "\n")
shutil.make_archive("../quant_results", "zip", OUT)
print("\n".join(lines))
print("\nA telecharger : benchmarking/quant_results.zip")

# Environnement - quantification Qwen/Qwen2.5-Coder-1.5B-Instruct

- GPU : Tesla T4
- Python : 3.13.16
- torch : 2.11.0+cu130
- CUDA : 13.0
- transformers : 5.20.2
- accelerate : 1.15.0
- bitsandbytes : 0.50.2
- huggingface_hub : 1.33.0
- datasets : 4.8.5
- pandas : 2.3.3

A telecharger : benchmarking/quant_results.zip
